In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData



In [2]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [3]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [4]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [5]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [6]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [7]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 36 points.


In [8]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9350
1    7255
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_96943/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [9]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [10]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [11]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [12]:
foldchange = 3

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=True)

Dropping 658 m/z features: ['136.0150' '137.0251' '138.0287' '139.0359' '154.0255' '155.0350'
 '156.0424' '159.0070' '160.0107' '163.0405' '165.0200' '172.0540'
 '173.0578' '176.0111' '177.0146' '178.0269' '179.0362' '181.0163'
 '187.0399' '189.0757' '190.0676' '191.0379' '192.9801' '195.0905'
 '196.0981' '196.9856' '197.1082' '197.9919' '199.0008' '199.0439'
 '199.9998' '200.0492' '201.0571' '202.0613' '202.1793' '203.0369'
 '203.0680' '203.1800' '204.0398' '210.9897' '211.0595' '213.0560'
 '213.9619' '214.0577' '214.9722' '215.0298' '216.0426' '217.0514'
 '218.0560' '219.0630' '219.9751' '220.0723' '220.9800' '226.1111'
 '227.0378' '228.0455' '229.0025' '229.0488' '230.0543' '231.0686'
 '232.0653' '233.0507' '233.0774' '235.9458' '239.0343' '240.0412'
 '241.0502' '242.0613' '243.0269' '243.0677' '244.0353' '245.0458'
 '246.0516' '247.0594' '248.0624' '249.0743' '251.0421' '254.0257'
 '255.0287' '256.0338' '257.0478' '258.0498' '259.0608' '259.2436'
 '260.0386' '261.0393' '262.0491' '

In [13]:
adata

AnnData object with n_obs × n_vars = 16605 × 342
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [14]:
adata.var_names

Index(['136.0608', '161.1338', '180.0717', '184.0720', '185.0686', '198.1210',
       '201.9372', '203.2236', '215.1771', '222.0327',
       ...
       '966.9248', '974.5565', '975.5518', '1494.1565', '1495.1523',
       '1520.1588', '1521.1633', '1522.1682', '1542.1475', '1544.1542'],
      dtype='object', length=342)

In [15]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the TIC-normalized intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Normalize by TIC
    if "TIC" not in adata.obs.columns:
        raise KeyError("TIC not found in adata.obs. Compute TIC before using this function.")
    tic = adata.obs["TIC"].values
    norm_intensities = intensities / tic
    norm_intensities[np.isnan(norm_intensities)] = 0  # Avoid NaNs from division by 0

    # Extract spatial coordinates
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    df = pd.DataFrame({"x": x, "y": y, "intensity": norm_intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale=[
            [0.0, "#000000"],  # black
            [0.10, "#000080"],  # navy
            [0.15, "#0000FF"],  # blue
            [0.30, "#8000FF"],  # purple-ish
            [0.45, "#FF0000"],  # red
            [0.60, "#FF8000"],  # orange
            [0.75, "#FFFF00"],  # yellow
            [1.0, "#FFFFFF"]   # white
        ],
        title=f"TIC-Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [16]:


# Ensure the image folder exists
os.makedirs(f"images_{foldchange}", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)
"""
# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_{foldchange}/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

"""

'\n# Modified plotting loop\nfor mz in selected_mz:\n    # Generate figure using your existing function\n    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig\n    \n    # Define filename\n    filename = f"images_{foldchange}/mz_{mz:.4f}.png"\n    \n    # Save the figure\n    pio.write_image(fig, filename)\n    print(f"Saved: {filename}")\n\n'

In [17]:
def mask_low_background_intensities(adata, fold_change=0.1):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [18]:
adata = mask_low_background_intensities(adata, fold_change=0.01)

In [19]:
def export_pixel_areas_to_csv_with_centroid(
    adata: AnnData,
    pixel_area: float = 100.0,
    output_csv: str = "pixel_area_summary.csv",
    x_key: str = "x",
    y_key: str = "y",
):
    """
    Calculates and exports per-m/z area, intensity sum, and centroid info to a CSV.

    Parameters:
        adata : AnnData
            AnnData object with .X, .obs["tissue"], .obs["x"], .obs["y"], and .var["mzs"] or .var_names
        pixel_area : float
            Area of a single pixel (e.g., in μm²)
        output_csv : str
            Output CSV filename
        x_key : str
            Key in adata.obs for x coordinates
        y_key : str
            Key in adata.obs for y coordinates
    """
    X = adata.X
    if sp.issparse(X):
        X = X.tocsr()

    if "mzs" in adata.var.columns:
        mz_values = adata.var["mzs"].values
    else:
        mz_values = pd.to_numeric(adata.var_names, errors="coerce")
        print("Warning: 'mzs' not found in var. Using var_names instead.")

    # Masks
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    x_coords = adata.obs[x_key].values
    y_coords = adata.obs[y_key].values

    # Initialize lists
    on_tissue_area = []
    non_tissue_area = []
    on_tissue_sum = []
    non_tissue_sum = []
    on_tissue_centroid_x = []
    on_tissue_centroid_y = []
    non_tissue_centroid_x = []
    non_tissue_centroid_y = []

    for mz_idx in range(X.shape[1]):
        intensities = X[:, mz_idx].toarray().flatten() if sp.issparse(X) else X[:, mz_idx]

        # Tissue and background
        tissue_values = intensities[tissue_mask.values]
        bg_values = intensities[background_mask.values]

        # Area
        on_tissue_area.append(np.sum(tissue_values > 0) * pixel_area)
        non_tissue_area.append(np.sum(bg_values > 0) * pixel_area)

        # Sum
        on_tissue_sum.append(np.sum(tissue_values))
        non_tissue_sum.append(np.sum(bg_values))

        # Centroid on tissue
        tx = x_coords[tissue_mask.values]
        ty = y_coords[tissue_mask.values]
        weight = tissue_values
        total = np.sum(weight)
        if total > 0:
            on_tissue_centroid_x.append(np.sum(tx * weight) / total)
            on_tissue_centroid_y.append(np.sum(ty * weight) / total)
        else:
            on_tissue_centroid_x.append(np.nan)
            on_tissue_centroid_y.append(np.nan)

        # Centroid on background
        bx = x_coords[background_mask.values]
        by = y_coords[background_mask.values]
        weight = bg_values
        total = np.sum(weight)
        if total > 0:
            non_tissue_centroid_x.append(np.sum(bx * weight) / total)
            non_tissue_centroid_y.append(np.sum(by * weight) / total)
        else:
            non_tissue_centroid_x.append(np.nan)
            non_tissue_centroid_y.append(np.nan)

    # DataFrame
    df = pd.DataFrame({
        "m/z": mz_values,
        "on_tissue_area": on_tissue_area,
        "non_tissue_area": non_tissue_area,
        "on_tissue_sum": on_tissue_sum,
        "non_tissue_sum": non_tissue_sum,
        "on_tissue_centroid_x": on_tissue_centroid_x,
        "on_tissue_centroid_y": on_tissue_centroid_y,
        "non_tissue_centroid_x": non_tissue_centroid_x,
        "non_tissue_centroid_y": non_tissue_centroid_y,
    })

    df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    return df

In [20]:
df = export_pixel_areas_to_csv_with_centroid(
    adata,
    pixel_area = 1.0,
    output_csv = "pixel_area_summary.csv",
    x_key = "x",
    y_key = "y",
)

Saved to pixel_area_summary.csv


In [21]:
df

,m/z,on_tissue_area,non_tissue_area,on_tissue_sum,non_tissue_sum,on_tissue_centroid_x,on_tissue_centroid_y,non_tissue_centroid_x,non_tissue_centroid_y
0,136.0608,7248.0,7988.0,2241834.0,1485472.0,63.557325,51.353482,64.466677,69.331504
1,161.1338,7116.0,8745.0,1071458.0,1764554.0,64.313635,58.478068,69.102606,92.004781
2,180.0717,7255.0,9349.0,3930515.0,6734531.0,63.878715,56.833429,68.278180,84.174816
3,184.0720,7255.0,8240.0,27214463.0,13078690.0,63.836410,55.370771,66.452270,66.998635
4,185.0686,7237.0,9332.0,1673455.0,2472849.0,64.073130,54.847721,69.430297,76.611636
...,...,...,...,...,...,...,...,...,...
337,1520.1588,7220.0,3839.0,702093.0,177486.0,63.828795,54.341413,67.877027,57.780715
338,1521.1633,7199.0,4580.0,625360.0,164861.0,63.754812,54.650510,68.459132,59.177677
339,1522.1682,7220.0,3573.0,677578.0,171106.0,63.785363,54.915591,67.251780,57.041401
340,1542.1475,7208.0,4511.0,552529.0,139278.0,63.507510,53.580762,65.993754,54.948642


In [22]:
import matplotlib.pyplot as plt
from skimage import measure

# Get contours at a constant value of 0.5 (between 0 and 1 in binary masks)
contours = measure.find_contours(mask, level=0.5)

# Plot the image with lower opacity and overlay the contour lines
plt.imshow(image, cmap='gray', alpha=0.3)  # Faded background
for contour in contours:
    plt.plot(contour[:, 1], contour[:, 0], linewidth=2, color='red')  # (x, y) = (col, row)

plt.axis('off')
plt.title('Polygon Margin Lines')
plt.show()

In [23]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=100,
    circle_size=50,
    figsize=(10, 8),
    show_margin=True,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using on-tissue squares and non-tissue circles.
    Optionally overlay the original image (faded) and polygon margin contours from mask.
    """
    required_cols = [
        "on_tissue_centroid_x", "on_tissue_centroid_y", "on_tissue_sum",
        "non_tissue_centroid_x", "non_tissue_centroid_y", "non_tissue_sum"
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    norm_on_tissue = plt.Normalize(vmin=df["on_tissue_sum"].min(), vmax=df["on_tissue_sum"].max())
    norm_non_tissue = plt.Normalize(vmin=df["non_tissue_sum"].min(), vmax=df["non_tissue_sum"].max())

    cmap_on_tissue_obj = plt.colormaps[cmap_on_tissue]
    cmap_non_tissue_obj = plt.colormaps[cmap_non_tissue]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image with alpha
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for _, row in df.iterrows():
        if not np.isnan(row["on_tissue_centroid_x"]) and not np.isnan(row["on_tissue_centroid_y"]):
            color = cmap_on_tissue_obj(norm_on_tissue(row["on_tissue_sum"]))
            square = patches.Rectangle(
                (row["on_tissue_centroid_x"] - square_size / 2, row["on_tissue_centroid_y"] - square_size / 2),
                square_size, square_size,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(square)

        if not np.isnan(row["non_tissue_centroid_x"]) and not np.isnan(row["non_tissue_centroid_y"]):
            color = cmap_non_tissue_obj(norm_non_tissue(row["non_tissue_sum"]))
            circle = patches.Circle(
                (row["non_tissue_centroid_x"], row["non_tissue_centroid_y"]),
                radius=circle_size / 2,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(circle)

    # Plot margin lines from mask
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Optional dashed margin lines based on centroid averages
    if show_margin:
        tissue_centroids = df[["on_tissue_centroid_x", "on_tissue_centroid_y"]].dropna()
        non_tissue_centroids = df[["non_tissue_centroid_x", "non_tissue_centroid_y"]].dropna()

        if not tissue_centroids.empty:
            avg_tissue_x = tissue_centroids["on_tissue_centroid_x"].mean()
            avg_tissue_y = tissue_centroids["on_tissue_centroid_y"].mean()
            ax.plot([avg_tissue_x - 100, avg_tissue_x + 100],
                    [avg_tissue_y - 100, avg_tissue_y + 100],
                    color="black", linewidth=margin_linewidth, linestyle="--")

        if not non_tissue_centroids.empty:
            avg_non_tissue_x = non_tissue_centroids["non_tissue_centroid_x"].mean()
            avg_non_tissue_y = non_tissue_centroids["non_tissue_centroid_y"].mean()
            ax.plot([avg_non_tissue_x - 100, avg_non_tissue_x + 100],
                    [avg_non_tissue_y - 100, avg_non_tissue_y + 100],
                    color="red", linewidth=margin_linewidth, linestyle="--")

    # Colorbars
    sm_on_tissue = plt.cm.ScalarMappable(cmap=cmap_on_tissue_obj, norm=norm_on_tissue)
    sm_on_tissue.set_array([])
    plt.colorbar(sm_on_tissue, ax=ax, label="On-Tissue Intensity Sum")

    sm_non_tissue = plt.cm.ScalarMappable(cmap=cmap_non_tissue_obj, norm=norm_non_tissue)
    sm_non_tissue.set_array([])
    plt.colorbar(sm_non_tissue, ax=ax, label="Non-Tissue Intensity Sum")

    # Final plot settings
    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids with Margin Lines and Image Background")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [24]:
plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=0.5,
    circle_size=0.5,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [25]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=100,
    circle_size=50,
    figsize=(10, 8),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using on-tissue squares and non-tissue circles.
    Optionally overlay the original image (faded) and polygon margin contours from mask.
    Intensities are log-scaled before color mapping.
    """
    required_cols = [
        "on_tissue_centroid_x", "on_tissue_centroid_y", "on_tissue_sum",
        "non_tissue_centroid_x", "non_tissue_centroid_y", "non_tissue_sum"
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Apply log1p to intensity values
    log_on_tissue = np.log1p(df["on_tissue_sum"])
    log_non_tissue = np.log1p(df["non_tissue_sum"])

    norm_on_tissue = plt.Normalize(vmin=log_on_tissue.min(), vmax=log_on_tissue.max())
    norm_non_tissue = plt.Normalize(vmin=log_non_tissue.min(), vmax=log_non_tissue.max())

    cmap_on_tissue_obj = plt.colormaps[cmap_on_tissue]
    cmap_non_tissue_obj = plt.colormaps[cmap_non_tissue]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image with alpha
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for i, row in df.iterrows():
        if not np.isnan(row["on_tissue_centroid_x"]) and not np.isnan(row["on_tissue_centroid_y"]):
            color = cmap_on_tissue_obj(norm_on_tissue(log_on_tissue.iloc[i]))
            square = patches.Rectangle(
                (row["on_tissue_centroid_x"] - square_size / 2, row["on_tissue_centroid_y"] - square_size / 2),
                square_size, square_size,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(square)

        if not np.isnan(row["non_tissue_centroid_x"]) and not np.isnan(row["non_tissue_centroid_y"]):
            color = cmap_non_tissue_obj(norm_non_tissue(log_non_tissue.iloc[i]))
            circle = patches.Circle(
                (row["non_tissue_centroid_x"], row["non_tissue_centroid_y"]),
                radius=circle_size / 2,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(circle)

    # Plot mask contours
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Colorbars
    sm_on_tissue = plt.cm.ScalarMappable(cmap=cmap_on_tissue_obj, norm=norm_on_tissue)
    sm_on_tissue.set_array([])
    plt.colorbar(sm_on_tissue, ax=ax, label="log(1 + On-Tissue Intensity Sum)")

    sm_non_tissue = plt.cm.ScalarMappable(cmap=cmap_non_tissue_obj, norm=norm_non_tissue)
    sm_non_tissue.set_array([])
    plt.colorbar(sm_non_tissue, ax=ax, label="log(1 + Non-Tissue Intensity Sum)")

    # Final plot settings
    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids with Mask Contours and Log-Scaled Intensities")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [26]:
plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    square_size=1,
    circle_size=1,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [27]:
import numpy as np
import pandas as pd
from scipy.stats import entropy
from libpysal.weights import DistanceBand
from libpysal.weights import KNN
from esda.moran import Moran
from tqdm import tqdm

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return 0
    p = intensities / intensities.sum()
    return entropy(p)


def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        print("Failed with error:", e)
        return np.nan

def analyze_scatter_wide_format(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values  # <-- Use "mzs"

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}
        for flag, region in zip([1, 0], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            row[f"spatial_entropy_{region}"] = compute_spatial_entropy(intensities)
            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)

        rows.append(row)

    return pd.DataFrame(rows)



In [28]:
"""
Moran's I and Spatial Entropy Analysis
This code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   
Moran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. 
Moran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.
moran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.
Spatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   
spatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.
The analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.
The DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.
"""

"\nMoran's I and Spatial Entropy Analysis\nThis code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   \nMoran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. \nMoran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.\nmoran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.\nSpatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   \nspatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.\nThe analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.\nThe DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.\n"

In [29]:
# Run it
scatter_wide_df = analyze_scatter_wide_format(adata, k_neighbors=8)

# Save to CSV (optional)
scatter_wide_df.to_csv("pixel_area_summary.csv", index=False)

# Preview
print(scatter_wide_df.head())

Processing m/z features: 100%|██████████| 342/342 [02:15<00:00,  2.52it/s]

         mz  spatial_entropy_tissue  moran_I_tissue  \
0  136.0608                8.721260        0.245253   
1  161.1338                8.693067        0.212371   
2  180.0717                8.792237        0.214215   
3  184.0720                8.730428        0.169584   
4  185.0686                8.702934        0.147569   

   spatial_entropy_background  moran_I_background  
0                    8.644276            0.496016  
1                    8.787657            0.709766  
2                    8.973465            0.746096  
3                    8.687199            0.535857  
4                    9.008362            0.348609  


In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame from the results
pixel_area_summary = pd.read_csv("pixel_area_summary.csv")

# Set up the figure size and layout
plt.figure(figsize=(8, 100))

# Plot heatmap for spatial entropy
sns.heatmap(scatter_wide_df[['spatial_entropy_tissue', 'spatial_entropy_background']].set_index(pixel_area_summary['mz']),
            cmap='coolwarm', annot=True, fmt=".2f", cbar=True, xticklabels=True, yticklabels=True)

plt.title("Spatial Entropy for Tissue vs. Background")
plt.xlabel("Region")
plt.ylabel("m/z")
plt.show()

In [31]:
# Plotting Spatial Entropy vs Moran's I for Tissue
plt.figure(figsize=(10, 6))

sns.scatterplot(data=pixel_area_summary, x="spatial_entropy_tissue", y="moran_I_tissue", hue="mz", palette="viridis")

plt.title("Spatial Entropy vs Moran's I for Tissue")
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.legend(title="m/z", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Plotting Spatial Entropy vs Moran's I for Background
plt.figure(figsize=(10, 6))

sns.scatterplot(data=pixel_area_summary, x="spatial_entropy_background", y="moran_I_background", hue="mz", palette="viridis")

plt.title("Spatial Entropy vs Moran's I for Background")
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.legend(title="m/z", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [32]:
# Plotting Histograms of Spatial Entropy
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(pixel_area_summary["spatial_entropy_tissue"], bins=30, kde=True, color='blue', label='Tissue')
sns.histplot(pixel_area_summary["spatial_entropy_background"], bins=30, kde=True, color='red', label='Background')
plt.title("Histogram of Spatial Entropy")
plt.xlabel("Spatial Entropy")
plt.ylabel("Frequency")
plt.legend()

# Plotting Histograms of Moran's I
plt.subplot(1, 2, 2)
sns.histplot(pixel_area_summary["moran_I_tissue"].dropna(), bins=30, kde=True, color='blue', label='Tissue')
sns.histplot(pixel_area_summary["moran_I_background"].dropna(), bins=30, kde=True, color='red', label='Background')
plt.title("Histogram of Moran's I")
plt.xlabel("Moran's I")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.show()

In [33]:
scatter_wide_df['moran_I_diff'] = scatter_wide_df['moran_I_tissue'] - scatter_wide_df['moran_I_background']
top_moran_I = scatter_wide_df.sort_values(by='moran_I_diff', ascending=False).head(10)

In [34]:
scatter_wide_df['entropy_diff'] = scatter_wide_df['spatial_entropy_background'] - scatter_wide_df['spatial_entropy_tissue']
top_entropy_diff = scatter_wide_df.sort_values(by='entropy_diff', ascending=False).head(10)

In [35]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(scatter_wide_df['spatial_entropy_tissue'], scatter_wide_df['moran_I_tissue'], label='Tissue', alpha=0.6)
plt.scatter(scatter_wide_df['spatial_entropy_background'], scatter_wide_df['moran_I_background'], label='Background', alpha=0.6, color='orange')
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [36]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(scatter_wide_df['spatial_entropy_tissue'], scatter_wide_df['moran_I_tissue'], label='Tissue', alpha=0.6)
plt.scatter(scatter_wide_df['spatial_entropy_background'], scatter_wide_df['moran_I_background'], label='Background', alpha=0.6, color='orange')

# Annotate each point with m/z value
for i, row in scatter_wide_df.iterrows():
    mz_label = f"{row['mz']:.1f}"  # format to 1 decimal
    plt.annotate(mz_label,
                 (row['spatial_entropy_tissue'], row['moran_I_tissue']),
                 textcoords="offset points",
                 xytext=(0, 4),
                 ha='center',
                 fontsize=6,
                 color='blue')
    plt.annotate(mz_label,
                 (row['spatial_entropy_background'], row['moran_I_background']),
                 textcoords="offset points",
                 xytext=(0, 4),
                 ha='center',
                 fontsize=6,
                 color='darkorange')

plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [37]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def compute_intensity_metrics(adata):
    tissue_mask = adata.obs['tissue'] == 1
    background_mask = adata.obs['tissue'] == 0

    X = adata.X
    if hasattr(X, "toarray"):  # If sparse matrix
        X = X.toarray()

    mean_tissue = np.mean(X[tissue_mask], axis=0)
    mean_background = np.mean(X[background_mask], axis=0)

    total_tissue = np.sum(X[tissue_mask], axis=0)
    total_background = np.sum(X[background_mask], axis=0)

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.divide(total_tissue, total_background)
        ratio[np.isnan(ratio)] = 0
        ratio[np.isinf(ratio)] = 0

    df = pd.DataFrame({
        "mz": adata.var["mzs"].values,
        "mean_intensity_tissue": mean_tissue,
        "mean_intensity_background": mean_background,
        "total_intensity_tissue": total_tissue,
        "total_intensity_background": total_background,
        "tissue_to_background_ratio": ratio
    })

    return df

In [38]:
intensity_df = compute_intensity_metrics(adata)


In [39]:
results_df = scatter_wide_df.merge(intensity_df, on="mz")


In [40]:
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI(df):
    plt.figure(figsize=(10, 6))
    
    scatter = plt.scatter(
        df["tissue_to_background_ratio"],
        df["moran_I_tissue"],
        c=df["spatial_entropy_tissue"],
        s=np.clip(df["total_intensity_tissue"] / 1e5, 10, 200),  # scale size
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title("m/z Features: Spatial Autocorrelation vs. Intensity")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [41]:
plot_intensity_vs_moransI(results_df)


In [42]:
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI_with_labels(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Compute a custom score if not provided
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)
    
    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    scatter = plt.scatter(
        df["tissue_to_background_ratio"],
        df["moran_I_tissue"],
        c=df["spatial_entropy_tissue"],
        s=np.clip(df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["tissue_to_background_ratio"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [43]:
"""
this function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.
It uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.
The function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.
The score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.
The function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  

"""

"\nthis function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.\nIt uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.\nThe function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.\nThe score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.\nThe function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  \n\n"

In [44]:
plot_intensity_vs_moransI_with_labels(results_df, top_n=50)


In [45]:
import numpy as np
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI_with_labels(df, top_n=10, score_fn=None, background_threshold=1.0):
    plt.figure(figsize=(10, 6))

    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Define foreground and background based on threshold
    foreground_df = df[df["tissue_to_background_ratio"] >= background_threshold]
    background_df = df[df["tissue_to_background_ratio"] < background_threshold]

    # Plot background with triangle markers
    bg_scatter = plt.scatter(
        background_df["tissue_to_background_ratio"],
        background_df["moran_I_tissue"],
        c=background_df["spatial_entropy_tissue"],
        s=np.clip(background_df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.6,
        edgecolor='gray',
        linewidth=0.5,
        marker='v',  # triangle down
        label='Background m/z'
    )

    # Plot foreground with circle markers
    fg_scatter = plt.scatter(
        foreground_df["tissue_to_background_ratio"],
        foreground_df["moran_I_tissue"],
        c=foreground_df["spatial_entropy_tissue"],
        s=np.clip(foreground_df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5,
        marker='o',
        label='Foreground m/z'
    )

    # Annotate top N points from full dataframe
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["tissue_to_background_ratio"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    cbar = plt.colorbar(fg_scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [46]:
plot_intensity_vs_moransI_with_labels(results_df, top_n=1000, background_threshold=1.0)


In [47]:
results_df

,mz,spatial_entropy_tissue,moran_I_tissue,spatial_entropy_background,moran_I_background,moran_I_diff,entropy_diff,mean_intensity_tissue,mean_intensity_background,total_intensity_tissue,total_intensity_background,tissue_to_background_ratio
0,136.0608,8.721260,0.245253,8.644276,0.496016,-0.250763,-0.076983,309.005376,158.874011,2241834.0,1485472.0,1.509173
1,161.1338,8.693067,0.212371,8.787657,0.709766,-0.497395,0.094590,147.685458,188.722353,1071458.0,1764554.0,0.607212
2,180.0717,8.792237,0.214215,8.973465,0.746096,-0.531881,0.181228,541.766368,720.270695,3930515.0,6734531.0,0.583636
3,184.0720,8.730428,0.169584,8.687199,0.535857,-0.366273,-0.043229,3751.132047,1398.790374,27214463.0,13078690.0,2.080825
4,185.0686,8.702934,0.147569,9.008362,0.348609,-0.201040,0.305428,230.662302,264.475829,1673455.0,2472849.0,0.676732
...,...,...,...,...,...,...,...,...,...,...,...,...
337,1520.1588,8.685201,0.291927,7.788131,0.584070,-0.292143,-0.897070,96.773673,18.982460,702093.0,177486.0,3.955766
338,1521.1633,8.672424,0.291561,7.910056,0.579810,-0.288249,-0.762368,86.197105,17.632193,625360.0,164861.0,3.793256
339,1522.1682,8.675217,0.293162,7.763029,0.581143,-0.287981,-0.912188,93.394624,18.300107,677578.0,171106.0,3.959990
340,1542.1475,8.693458,0.240210,8.029168,0.500676,-0.260466,-0.664290,76.158374,14.896043,552529.0,139278.0,3.967095


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_top_mz_by_background_moransI(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default scoring function
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Plot top N as circles
    scatter = plt.scatter(
        top_rows["moran_I_background"],
        top_rows["moran_I_tissue"],
        c=top_rows["spatial_entropy_tissue"],
        s=np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.9,
        edgecolor='black',
        linewidth=0.6,
        marker='o',
        label='Top m/z'
    )

    # Annotate m/z values
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["moran_I_background"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    # Colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")

    # Size legend for circle areas (intensity)
    for val in [1e5, 3e5, 5e5]:
        plt.scatter([], [], s=val / 1e5, c='gray', alpha=0.5,
                    label=f"{int(val):,} total intensity")

    # Plot aesthetics
    plt.xlabel("Moran's I (Background)")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    plt.legend(title="Circle Size: Intensity", loc='lower right', frameon=True)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_mz_by_background_moransI(results_df, top_n=1000,score_fn=None)

NameError: name 'results_df' is not defined